# 14.04 (Colab) · QLoRA fine-tune of the Orbita Cloud ticket router

**Module 14 · Local Models & Fine-Tuning** · Part 2 (Pro) · elective · runs on **Google Colab**, not on your laptop

| | |
|---|---|
| ⏱️ Estimated time | 30–45 min (about 10–15 min of GPU time) |
| 📋 Prerequisites | Lesson 14.03 (dataset, baselines), a Google account |
| 🔑 Keys | none · optional `HF_TOKEN` (Colab secret) to push the GGUF to your Hub account |
| 🧠 Runtime | **Runtime → Change runtime type → T4 GPU** (free tier) |
| ✅ Docs verified | 2026-10-16 |

**How to run it:** in Colab, *File → Upload notebook* and pick this file (`modules/14-local-models-and-fine-tuning/colab/colab-qlora-router.ipynb` in your clone). Switch to a **T4 GPU**, then *Runtime → Run all*. When asked, upload `data/samples/local/tickets.jsonl` from your clone. At the end you download two files: `qlora-predictions.json` (small) and a GGUF model (about 1.1 GB).

Outside Colab (or without a GPU) every cell prints `⏭️ Skipped`, so the course's Restart & Run All check still passes.

In [ ]:
# Step 1 · Where am I? Colab + GPU → LIVE.
import importlib.util
import json
import os
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
HAVE_GPU = False
if importlib.util.find_spec("torch"):
    import torch
    HAVE_GPU = torch.cuda.is_available()
LIVE = IN_COLAB and HAVE_GPU
print("Colab:", IN_COLAB, "| GPU:", torch.cuda.get_device_name(0) if HAVE_GPU else "none", "| LIVE:", LIVE)
if IN_COLAB and not HAVE_GPU:
    print("⚠️ No GPU: Runtime → Change runtime type → T4 GPU, then Runtime → Run all.")

In [ ]:
# Step 2 · Install Unsloth (2–3 min on Colab). VERIFIED 2026-10-16: unsloth 2026.10.6.
if LIVE:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "unsloth>=2026.10.6"], check=True)
    print("✅ unsloth installed")
else:
    print("⏭️ Skipped: run this notebook on Google Colab with a T4 GPU (see the first cell).")

In [ ]:
# Step 3 · The tickets: upload data/samples/local/tickets.jsonl (or find it in a local clone).
SYSTEM_PROMPT = 'You route Orbita Cloud support tickets. Answer with exactly one queue name: billing, outage, account_access, bug, how_to, feature_request. If a ticket has two intents, pick the one the team must act on first.'
QUEUES = ("billing", "outage", "account_access", "bug", "how_to", "feature_request")

path = next((p for p in [Path("tickets.jsonl"), *(d / "data/samples/local/tickets.jsonl" for d in [Path.cwd(), *Path.cwd().parents])]
             if p.is_file()), None)
if path is None and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()                       # pick tickets.jsonl
    path = Path(next(iter(uploaded)))
TICKETS = [json.loads(line) for line in path.open(encoding="utf-8")] if path else []
TRAIN = [t for t in TICKETS if t["split"] == "train"]
TEST = [t for t in TICKETS if t["split"] == "test"]
print(len(TICKETS), "tickets |", len(TRAIN), "train |", len(TEST), "test" if TICKETS else "(no file: upload tickets.jsonl)")

The prompt is the same `SYSTEM_PROMPT` as `mastery.finetune` (Lesson 14.03): the tuned model must be served later with **exactly** this system message.

In [ ]:
# Step 4 · Load Qwen3-1.7B in 4-bit and add LoRA adapters (QLoRA). ≈ 1.5 GB download, ~3 GB of the T4's 15 GB.
BASE = "unsloth/Qwen3-1.7B"     # ← "unsloth/Qwen3-4B" also fits a T4 (slower, usually a bit better)
if LIVE and TRAIN:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE, max_seq_length=512, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth")
    model.print_trainable_parameters()
else:
    print("⏭️ Skipped: run this notebook on Google Colab with a T4 GPU (see the first cell).")

In [ ]:
# Step 5 · Train 2 epochs on the 594 training tickets, loss on the answer only (≈ 3–6 min on a T4).
if LIVE and TRAIN:
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer
    from unsloth.chat_templates import train_on_responses_only

    def as_text(t):
        messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": t["text"]},
                    {"role": "assistant", "content": t["queue"]}]
        return {"text": tokenizer.apply_chat_template(messages, tokenize=False, enable_thinking=False)}

    dataset = Dataset.from_list([as_text(t) for t in TRAIN])
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=dataset, args=SFTConfig(
        dataset_text_field="text", per_device_train_batch_size=8, gradient_accumulation_steps=1, num_train_epochs=2,
        learning_rate=2e-4, warmup_steps=10, lr_scheduler_type="linear", optim="adamw_8bit", weight_decay=0.0,
        logging_steps=10, seed=3407, output_dir="outputs", report_to="none"))
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
    start = time.perf_counter()
    stats = trainer.train()
    TRAIN_SECONDS = round(time.perf_counter() - start)
    print(f"trained in {TRAIN_SECONDS} s, final loss {stats.training_loss:.3f}")
else:
    print("⏭️ Skipped: run this notebook on Google Colab with a T4 GPU (see the first cell).")

In [ ]:
# Step 6 · Label the 234 test tickets (greedy, ≤ 6 new tokens) and save qlora-predictions.json.
if LIVE and TRAIN:
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    labels, start = {}, time.perf_counter()
    for b in range(0, len(TEST), 32):
        chunk = TEST[b:b + 32]
        prompts = [tokenizer.apply_chat_template([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": t["text"]}],
                                                 tokenize=False, add_generation_prompt=True, enable_thinking=False) for t in chunk]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = model.generate(**enc, max_new_tokens=6, do_sample=False)
        for t, o in zip(chunk, out):
            words = tokenizer.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).split()
            labels[t["id"]] = words[0] if words and words[0] in QUEUES else "error"
    right = sum(labels[t["id"]] == t["queue"] for t in TEST)
    print(f"test accuracy {right}/{len(TEST)} = {right / len(TEST):.1%} in {time.perf_counter() - start:.0f} s")
    record = {"model": f"qlora:{BASE} (r=16, 2 epochs)", "recorded": time.strftime("%Y-%m-%d"), "split": "test",
              "method": "QLoRA 4-bit, r=16 alpha=16 all linear layers, 2 epochs, lr 2e-4, batch 8, answer-only loss; greedy",
              "hardware": torch.cuda.get_device_name(0) + " (Colab)", "train_seconds": TRAIN_SECONDS,
              "train_examples": len(TRAIN), "labels": dict(sorted(labels.items()))}
    Path("qlora-predictions.json").write_text(json.dumps(record, indent=1) + "\n", encoding="utf-8")
    from google.colab import files
    files.download("qlora-predictions.json")       # → put it in data/outputs/m14/ in your clone
else:
    print("⏭️ Skipped: run this notebook on Google Colab with a T4 GPU (see the first cell).")

In [ ]:
# Step 7 · Merge the adapters and export a Q4_K_M GGUF for Ollama (≈ 5 min, ≈ 1.1 GB). Optional push to your Hub account.
PUSH_TO_HUB = ""        # ← e.g. "your-hf-username/orbita-router-GGUF" (needs the HF_TOKEN Colab secret); "" = download
if LIVE and TRAIN:
    model.save_pretrained_gguf("orbita-router", tokenizer, quantization_method="q4_k_m")
    ggufs = sorted(Path(".").rglob("*.gguf"))
    print("GGUF files:", [f"{p} ({p.stat().st_size / 1e9:.2f} GB)" for p in ggufs])
    if PUSH_TO_HUB:
        from google.colab import userdata
        model.push_to_hub_gguf(PUSH_TO_HUB, tokenizer, quantization_method="q4_k_m", token=userdata.get("HF_TOKEN"))
        print(f"✅ pushed: ollama pull hf.co/{PUSH_TO_HUB}:Q4_K_M")
    else:
        from google.colab import files
        files.download(str(next(p for p in ggufs if "Q4_K_M" in p.name.upper())))
else:
    print("⏭️ Skipped: run this notebook on Google Colab with a T4 GPU (see the first cell).")

**Next:** back in Lesson 14.04 on your laptop, put `qlora-predictions.json` in `data/outputs/m14/`, then serve the GGUF with Ollama (Part 4 of the lesson). Close the Colab tab when done: *Runtime → Disconnect and delete runtime* frees your GPU quota.